# Stock Price Forecasting: Classical ML, LSTM, Mamba, and Regime-Aware Models

This notebook is the **structured market-data branch** of the project. It benchmarks next-trading-day **price regression** on AAPL and TSLA using:

- Naive persistence baseline
- XGBoost
- Gaussian HMM + Decision Tree
- LSTM
- Mamba / selective state-space model
- Multi-scale LSTM hybrid

## Experimental principles

1. **Chronological evaluation only** — no random train/test split.
2. **No look-ahead feature filling** — rolling features use past observations only; incomplete warm-up rows are dropped.
3. **Real Mamba only** — if `mamba_ssm` is unavailable, the notebook stops instead of silently replacing Mamba with an identity layer.
4. **Strong baseline included** — the previous close is evaluated as a Naive predictor.
5. **Reproducibility** — random seeds are controlled and daily predictions can be exported.

> The news / LLM branch (Gemini, FinBERT, Point-in-Time alignment, Time Decay) is maintained separately in the StockLLM repository. It is intentionally excluded here so the market-model benchmark remains auditable and leakage-free.

## 1. Environment

Recommended: Google Colab with GPU. `mamba-ssm` may require a compatible PyTorch/CUDA environment.

In [ ]:
# Core dependencies
%pip install -q yfinance scikit-learn matplotlib xgboost hmmlearn torch

# Mamba dependencies. If installation fails, resolve the environment first;
# this notebook deliberately does NOT use a fake fallback implementation.
%pip install -q causal-conv1d>=1.2.0 mamba-ssm

## 2. Imports and reproducibility

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import xgboost as xgb

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")

try:
    from mamba_ssm import Mamba
except ImportError as exc:
    raise ImportError(
        "mamba_ssm is required for the Mamba experiment. "
        "Install a compatible version instead of using a fallback layer."
    ) from exc

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

## 3. Market data and causal feature engineering

The target day is downloaded only so its realized close can be used for evaluation. All model inputs are cut off strictly before the target day.

Rolling features are **not back-filled**. Back-filling can inject later observations into early rows, so warm-up rows are dropped instead.

In [ ]:
def download_market_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Download daily OHLCV data, inclusive of end_date for evaluation."""
    end_exclusive = (pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = yf.download(
        ticker,
        start=start_date,
        end=end_exclusive,
        auto_adjust=False,
        progress=False,
    )
    if df.empty:
        raise ValueError(f"No data returned for {ticker}")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[["Open", "High", "Low", "Close", "Volume"]].copy()


def add_market_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["MA5"] = out["Close"].rolling(5).mean()
    out["MA20"] = out["Close"].rolling(20).mean()
    out["Volatility_10"] = out["Close"].rolling(10).std()
    out["Return"] = out["Close"].pct_change()

    # Heuristic multi-scale decomposition used by the Hybrid model.
    # This is NOT MEMD; it is a lightweight approximation inspired by
    # decomposition-based forecasting literature.
    ma3 = out["Close"].rolling(3).mean()
    ma10 = out["Close"].rolling(10).mean()
    out["IMF1_HighFreq"] = out["Close"] - ma3
    out["IMF2_MidFreq"] = ma3 - ma10
    out["Trend_LowFreq"] = ma10

    return out.dropna().copy()

BASE_FEATURES = [
    "Open", "High", "Low", "Close", "Volume",
    "MA5", "MA20", "Volatility_10",
]
HYBRID_FEATURES = BASE_FEATURES + [
    "IMF1_HighFreq", "IMF2_MidFreq", "Trend_LowFreq",
]

## 4. Sequence models

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64, num_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :]))


class StockMamba(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int = 64,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.mamba(x)
        return self.head(self.dropout(x[:, -1, :]))

## 5. Training helpers

The sequence target is scaled independently from the feature matrix rather than relying on a hard-coded column position.

In [ ]:
def make_sequences(X: np.ndarray, y: np.ndarray, seq_length: int):
    xs, ys = [], []
    for i in range(len(X) - seq_length):
        xs.append(X[i : i + seq_length])
        ys.append(y[i + seq_length])
    return np.asarray(xs), np.asarray(ys).reshape(-1, 1)


def train_sequence_model(
    model: nn.Module,
    train_df: pd.DataFrame,
    features: list[str],
    seq_length: int = 60,
    epochs: int = 15,
    seed: int = 42,
    device: torch.device = DEVICE,
) -> float:
    set_seed(seed)

    x_scaler = MinMaxScaler()
    y_scaler = MinMaxScaler()
    X_scaled = x_scaler.fit_transform(train_df[features])
    y_scaled = y_scaler.fit_transform(train_df[["Close"]]).ravel()

    X_seq, y_seq = make_sequences(X_scaled, y_scaled, seq_length)
    if len(X_seq) == 0:
        raise ValueError("Not enough history to create training sequences")

    loader = DataLoader(
        TensorDataset(torch.tensor(X_seq, dtype=torch.float32),
                      torch.tensor(y_seq, dtype=torch.float32)),
        batch_size=32,
        shuffle=False,
    )

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    last_sequence = torch.tensor(
        X_scaled[-seq_length:], dtype=torch.float32, device=device
    ).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        pred_scaled = model(last_sequence).cpu().numpy()
    return float(y_scaler.inverse_transform(pred_scaled)[0, 0])

## 6. One-day prediction

For a target trading day `t`, every model receives only rows strictly before `t`. The realized close at `t` is used only after prediction for error calculation.

In [ ]:
def predict_one_day(
    ticker: str,
    target_date: str,
    seq_length: int = 60,
    epochs: int = 15,
    seed: int = 42,
    lookback_days: int = 730,
):
    target_ts = pd.Timestamp(target_date)
    start_date = (target_ts - pd.Timedelta(days=lookback_days)).strftime("%Y-%m-%d")

    raw = download_market_data(ticker, start_date, target_date)
    df = add_market_features(raw)

    if target_ts not in df.index:
        return None  # holiday / non-trading day / insufficient rolling warm-up

    target_pos = df.index.get_loc(target_ts)
    train_df = df.iloc[:target_pos].copy()
    if len(train_df) <= seq_length + 2:
        return None

    actual = float(df.loc[target_ts, "Close"])

    # 1) Naive persistence
    pred_naive = float(train_df.iloc[-1]["Close"])

    # 2) XGBoost: target is next observed close within the training window.
    xgb_df = train_df.copy()
    xgb_df["Target"] = xgb_df["Close"].shift(-1)
    xgb_train = xgb_df.dropna(subset=["Target"])
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        random_state=seed,
        objective="reg:squarederror",
    )
    xgb_model.fit(xgb_train[BASE_FEATURES], xgb_train["Target"])
    pred_xgb = float(xgb_model.predict(xgb_df.iloc[[-1]][BASE_FEATURES])[0])

    # 3) Gaussian HMM + Decision Tree.
    # HMM extracts latent regime probabilities; DT predicts the next close.
    hmm_input = train_df[["Return", "Volatility_10"]].to_numpy()
    hmm = GaussianHMM(
        n_components=3,
        covariance_type="diag",
        n_iter=100,
        random_state=seed,
    )
    hmm.fit(hmm_input)
    state_probs = hmm.predict_proba(hmm_input)

    dt_df = train_df.copy()
    for j in range(3):
        dt_df[f"State_{j}"] = state_probs[:, j]
    dt_df["Target"] = dt_df["Close"].shift(-1)
    dt_train = dt_df.dropna(subset=["Target"])
    dt_features = BASE_FEATURES + ["State_0", "State_1", "State_2"]

    dt_model = DecisionTreeRegressor(max_depth=5, random_state=seed)
    dt_model.fit(dt_train[dt_features], dt_train["Target"])
    pred_hmm_dt = float(dt_model.predict(dt_df.iloc[[-1]][dt_features])[0])

    # 4) LSTM
    pred_lstm = train_sequence_model(
        StockLSTM(len(BASE_FEATURES)), train_df, BASE_FEATURES,
        seq_length=seq_length, epochs=epochs, seed=seed,
    )

    # 5) Real Mamba (no identity fallback)
    pred_mamba = train_sequence_model(
        StockMamba(len(BASE_FEATURES)), train_df, BASE_FEATURES,
        seq_length=seq_length, epochs=epochs, seed=seed,
    )

    # 6) Multi-scale LSTM hybrid (heuristic decomposition, not MEMD)
    pred_hybrid = train_sequence_model(
        StockLSTM(len(HYBRID_FEATURES)), train_df, HYBRID_FEATURES,
        seq_length=seq_length, epochs=epochs, seed=seed,
    )

    return {
        "Ticker": ticker,
        "Date": target_date,
        "Actual": actual,
        "Naive": pred_naive,
        "XGBoost": pred_xgb,
        "HMM+DT": pred_hmm_dt,
        "LSTM": pred_lstm,
        "Mamba": pred_mamba,
        "Hybrid": pred_hybrid,
    }

## 7. Chronological backtest and MAPE summary

In [ ]:
MODEL_NAMES = ["Naive", "XGBoost", "HMM+DT", "LSTM", "Mamba", "Hybrid"]


def mape(actual, pred):
    return abs(pred - actual) / abs(actual) * 100.0


def run_backtest(
    tickers=("AAPL", "TSLA"),
    target_month="2025-11",
    seq_length=60,
    epochs=15,
    seed=42,
):
    month_start = pd.Timestamp(f"{target_month}-01")
    month_end = month_start + pd.offsets.MonthEnd(0)
    candidate_dates = pd.bdate_range(month_start, month_end)

    rows = []
    for ticker in tickers:
        print(f"\n=== {ticker} ===")
        for ts in candidate_dates:
            date_str = ts.strftime("%Y-%m-%d")
            try:
                pred = predict_one_day(
                    ticker,
                    date_str,
                    seq_length=seq_length,
                    epochs=epochs,
                    seed=seed,
                )
            except Exception as exc:
                print(f"{date_str}: failed -> {exc}")
                continue

            if pred is None:
                continue

            for model in MODEL_NAMES:
                pred[f"Err_{model}(%)"] = mape(pred["Actual"], pred[model])
            rows.append(pred)
            print(f"{date_str}: done")

    daily = pd.DataFrame(rows)
    if daily.empty:
        raise RuntimeError("No valid predictions were generated")

    err_cols = [f"Err_{m}(%)" for m in MODEL_NAMES]
    summary = daily.groupby("Ticker")[err_cols].mean().reset_index()

    daily.to_csv(OUTPUT_DIR / "market_backtest_daily.csv", index=False)
    summary.to_csv(OUTPUT_DIR / "market_backtest_mape_summary.csv", index=False)
    return daily, summary


daily_results, summary = run_backtest(
    tickers=("AAPL", "TSLA"),
    target_month="2025-11",
    epochs=15,
    seed=42,
)

display(summary.round(4))

## 8. Plot results

In [ ]:
plot_df = summary.set_index("Ticker").copy()
plot_df.columns = [c.replace("Err_", "").replace("(%)", "") for c in plot_df.columns]

ax = plot_df.plot(kind="bar", figsize=(12, 6))
ax.set_title("Next-Day Price Forecasting Error — 2025-11")
ax.set_ylabel("MAPE (%)")
ax.set_xlabel("Ticker")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "market_model_mape.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Optional: random-seed stability check

Neural models can vary across initialization. This cell tests LSTM, Mamba, and the multi-scale Hybrid on the same target date under multiple seeds. The result should be interpreted as a **stability diagnostic**, not as a second test set.

In [ ]:
def seed_stability_test(
    ticker="AAPL",
    target_date="2025-11-20",
    seeds=(42, 43, 44, 45, 46),
    epochs=15,
):
    rows = []
    for seed in seeds:
        pred = predict_one_day(ticker, target_date, epochs=epochs, seed=seed)
        if pred is None:
            continue
        for model in ["LSTM", "Mamba", "Hybrid"]:
            rows.append({
                "Seed": seed,
                "Model": model,
                "MAPE(%)": mape(pred["Actual"], pred[model]),
            })

    detail = pd.DataFrame(rows)
    summary_seed = (
        detail.groupby("Model")["MAPE(%)"]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "MAPE_Mean(%)", "std": "MAPE_Std(%)"})
    )
    detail.to_csv(OUTPUT_DIR / "seed_stability_detail.csv", index=False)
    summary_seed.to_csv(OUTPUT_DIR / "seed_stability_summary.csv", index=False)
    return detail, summary_seed

# Uncomment when you want the additional compute cost:
# seed_detail, seed_summary = seed_stability_test()
# display(seed_summary.round(4))

## 10. Interpretation and limitations

- The benchmark evaluates **one-step-ahead price regression**, not trading profitability.
- Mamba is evaluated as a real `mamba_ssm` model; the notebook intentionally fails if the package is unavailable.
- The HMM branch is a **Gaussian HMM + Decision Tree** implementation. It is inspired by regime-aware hybrid literature but is not a full HHMM + rough-set reproduction.
- The multi-scale Hybrid uses heuristic moving-average decomposition and should not be described as MEMD.
- AAPL/TSLA and a one-month test window are not sufficient to claim universal model superiority. More tickers, longer held-out periods, and multi-seed runs are appropriate extensions.
- News/LLM experiments are kept in the companion StockLLM repository because they use a different target (next-day direction) and a stricter Point-in-Time text pipeline.

### Why this notebook was cleaned

Earlier exploratory cells mixed market forecasting with prototype sentiment code and demonstration plots. Those cells were removed from the public research notebook because:

1. an early simulated sentiment feature used `future_return`, which is look-ahead leakage and is **not part of the formal experiment**;
2. a previous Mamba import fallback silently behaved like an identity layer, which could mislabel results as Mamba;
3. simulated prediction trajectories are inappropriate as research evidence.

The public version therefore keeps only the auditable market-model benchmark.